In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 46.9 MB/s eta 0:00:00


In [2]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets

print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

✅ CUDA available: True
✅ GPU: Tesla T4


In [3]:
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
OUTPUT_DIR = "./codellama-coding-chat"

MAX_SEQ_LENGTH = 1024
CODE_SAMPLES = 15000
CHAT_SAMPLES = 6000

print(f"📁 Output: {OUTPUT_DIR}")
print(f"📊 Samples: {CODE_SAMPLES} code + {CHAT_SAMPLES} chat = {CODE_SAMPLES + CHAT_SAMPLES} total")

📁 Output: ./codellama-coding-chat
📊 Samples: 15000 code + 6000 chat = 21000 total


In [4]:
print("\n🚀 Loading datasets to CPU...")

print("Loading code dataset...")
code_dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
code_dataset = code_dataset.shuffle(seed=42).select(range(min(CODE_SAMPLES, len(code_dataset))))
print(f"✅ Code: {len(code_dataset)} examples")

print("Loading chat dataset...")
chat_dataset = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train")
chat_dataset = chat_dataset.shuffle(seed=42).select(range(min(CHAT_SAMPLES, len(chat_dataset))))
print(f"✅ Chat: {len(chat_dataset)} examples")


🚀 Loading datasets to CPU...
Loading code dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

✅ Code: 15000 examples
Loading chat dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/127M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44245 [00:00<?, ? examples/s]

✅ Chat: 6000 examples


In [6]:
def format_code(example):
    instruction = example.get('prompt', example.get('instruction', ''))
    input_text = example.get('input', '')
    output = example.get('completion', example.get('output', ''))

    if input_text:
        prompt = f"{instruction}\n\nInput: {input_text}"
    else:
        prompt = instruction

    return f"[INST] {prompt.strip()} [/INST] {output.strip()}"

def format_chat(example):
    prompt = example.get('prompt', example.get('question', example.get('instruction', '')))
    response = example.get('chosen', example.get('response', example.get('answer', example.get('output', ''))))

    if isinstance(response, list):
        response = response[0] if response else ""

    return f"[INST] {prompt.strip()} [/INST] {str(response).strip()}"

print("✅ Format functions ready")

✅ Format functions ready


In [7]:
print("\n⚙️  Formatting datasets (CPU processing)...")

code_formatted = code_dataset.map(
    lambda x: {"text": format_code(x)},
    remove_columns=code_dataset.column_names,
    desc="Formatting code"
)

chat_formatted = chat_dataset.map(
    lambda x: {"text": format_chat(x)},
    remove_columns=chat_dataset.column_names,
    desc="Formatting chat"
)

full_dataset = concatenate_datasets([code_formatted, chat_formatted])
full_dataset = full_dataset.shuffle(seed=42)

print(f"✅ Total: {len(full_dataset)} examples")
print(f"📖 Sample: {full_dataset[0]['text'][:150]}...")


⚙️  Formatting datasets (CPU processing)...


Formatting code:   0%|          | 0/15000 [00:00<?, ? examples/s]

Formatting chat:   0%|          | 0/6000 [00:00<?, ? examples/s]

✅ Total: 21000 examples
📖 Sample: [INST] Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Interpret the resul...


In [8]:
print("\n🚀 Loading model to GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded!")
print(f"📊 Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"💾 GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


🚀 Loading model to GPU...


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded!
📊 Trainable: 79,953,920 (2.23%)
💾 GPU Memory: 4.73 GB


In [9]:
print("\n🔍 Checking for checkpoints...")

class ProgressCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        with open(os.path.join(args.output_dir, "latest_checkpoint.txt"), "w") as f:
            f.write(f"{state.global_step}")
        print(f"💾 Saved checkpoint at step {state.global_step}")

last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))
        last_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])
        print(f"⏮️  Resuming from: {last_checkpoint}")
    else:
        print("⏳ No checkpoints found - starting fresh")
else:
    print("⏳ Output dir not found - starting fresh")
    os.makedirs(OUTPUT_DIR, exist_ok=True)


🔍 Checking for checkpoints...
⏳ Output dir not found - starting fresh


In [11]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    max_grad_norm=0.3,
    warmup_steps=0.05,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [15]:
print("\n🔧 Setting up trainer...")

from transformers import Trainer, DataCollatorForLanguageModeling

# Tokenize the dataset first
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length"
    )

print("Tokenizing dataset...")
tokenized_dataset = full_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=full_dataset.column_names
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[ProgressCallback],
)

print("✅ Trainer ready!")
print(f"🎯 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📈 Total steps: ~{len(tokenized_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")


🔧 Setting up trainer...
Tokenizing dataset...


Map:   0%|          | 0/21000 [00:00<?, ? examples/s]

✅ Trainer ready!
🎯 Effective batch size: 8
📈 Total steps: ~2625


In [16]:
print("\n🔥 STARTING TRAINING!")
print(f"💻 GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB / 15.0 GB")
print("=" * 50)

trainer.train(resume_from_checkpoint=last_checkpoint)

print("=" * 50)
print("✅ Training complete!")


🔥 STARTING TRAINING!
💻 GPU: Tesla T4
💾 GPU Memory: 4.73 GB / 15.0 GB


Step,Training Loss
25,0.997448
50,0.538427
75,0.427842
100,0.440777
125,0.491159
150,0.453752
175,0.398109
200,0.431024
225,0.519200
250,0.441863


💾 Saved checkpoint at step 250


KeyboardInterrupt: 

In [18]:
# COPY CHECKPOINT TO DRIVE
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Copy checkpoint-250 to Drive
source = "./codellama-coding-chat/checkpoint-250"
dest = "/content/drive/MyDrive/codellama-coding-chat/checkpoint-250"

os.makedirs(dest, exist_ok=True)
shutil.copytree(source, dest, dirs_exist_ok=True)

print(f"✅ Checkpoint copied to Google Drive!")
print(f"📁 Location: {dest}")

Mounted at /content/drive
✅ Checkpoint copied to Google Drive!
📁 Location: /content/drive/MyDrive/codellama-coding-chat/checkpoint-250


In [19]:
# SAVE STATE FOR GITHUB
import json
from datetime import datetime

state = {
    "last_step": 250,
    "checkpoint_location": "Google Drive: /MyDrive/codellama-coding-chat/checkpoint-250",
    "status": "paused_at_250",
    "timestamp": datetime.now().isoformat()
}

with open("training_state.json", "w") as f:
    json.dump(state, f)

print("✅ State saved!")
print("\n📤 Push to GitHub:")
print("!git add training_state.json && git commit -m 'Pause at 250' && git push")

✅ State saved!

📤 Push to GitHub:
!git add training_state.json && git commit -m 'Pause at 250' && git push


In [20]:
# CHECK EVERYTHING - Model & Checkpoint Status
import os
import json

print("=" * 60)
print("🔍 CHECKING SAVED FILES")
print("=" * 60)

# 1. CHECK LOCAL CHECKPOINT
checkpoint_dir = "./codellama-coding-chat"
checkpoint_250 = os.path.join(checkpoint_dir, "checkpoint-250")

print(f"\n📁 Local Checkpoint:")
print(f"   Path: {checkpoint_250}")
if os.path.exists(checkpoint_250):
    files = os.listdir(checkpoint_250)
    size = sum(os.path.getsize(os.path.join(dirpath, f))
               for dirpath, dirnames, filenames in os.walk(checkpoint_250)
               for f in filenames) / (1024**3)
    print(f"   Status: ✅ EXISTS")
    print(f"   Size: {size:.2f} GB")
    print(f"   Files: {len(files)} items")
    key_files = [f for f in files if 'adapter' in f or 'model' in f]
    print(f"   Model files: {key_files[:3]}...")
else:
    print(f"   Status: ❌ NOT FOUND")

# 2. CHECK GOOGLE DRIVE
drive_path = "/content/drive/MyDrive/codellama-coding-chat/checkpoint-250"
print(f"\n💾 Google Drive Checkpoint:")
print(f"   Path: {drive_path}")
if os.path.exists(drive_path):
    drive_size = sum(os.path.getsize(os.path.join(dirpath, f))
                     for dirpath, dirnames, filenames in os.walk(drive_path)
                     for f in filenames) / (1024**3)
    print(f"   Status: ✅ EXISTS")
    print(f"   Size: {drive_size:.2f} GB")
else:
    print(f"   Status: ❌ NOT FOUND (run Drive copy first)")

# 3. CHECK FINAL MODEL
final_model = "./codellama-coding-chat-final"
print(f"\n🎯 Final Model:")
print(f"   Path: {final_model}")
if os.path.exists(final_model):
    final_files = os.listdir(final_model)
    final_size = sum(os.path.getsize(os.path.join(dirpath, f))
                    for dirpath, dirnames, filenames in os.walk(final_model)
                    for f in filenames) / (1024**3)
    print(f"   Status: ✅ EXISTS")
    print(f"   Size: {final_size:.2f} GB")
    print(f"   Files: {final_files[:5]}")
else:
    print(f"   Status: ❌ NOT FOUND")

# 4. CHECK STATE FILE
print(f"\n📄 GitHub State File:")
if os.path.exists("training_state.json"):
    with open("training_state.json") as f:
        state = json.load(f)
    print(f"   Status: ✅ EXISTS")
    print(f"   Step: {state.get('last_step', 'unknown')}")
    print(f"   Status: {state.get('status', 'unknown')}")
else:
    print(f"   Status: ❌ NOT FOUND")

# 5. SUMMARY
print("\n" + "=" * 60)
print("📊 SUMMARY")
print("=" * 60)

checks = {
    "Local Checkpoint (250)": os.path.exists(checkpoint_250),
    "Drive Checkpoint": os.path.exists(drive_path),
    "Final Model": os.path.exists(final_model),
    "State JSON": os.path.exists("training_state.json")
}

all_good = all(checks.values())

for name, status in checks.items():
    icon = "✅" if status else "❌"
    print(f"   {icon} {name}")

print("=" * 60)
if all_good:
    print("🎉 EVERYTHING SAVED! Ready to resume tomorrow.")
else:
    print("⚠️  SOME FILES MISSING - Check above.")
print("=" * 60)

🔍 CHECKING SAVED FILES

📁 Local Checkpoint:
   Path: ./codellama-coding-chat/checkpoint-250
   Status: ✅ EXISTS
   Size: 0.45 GB
   Files: 12 items
   Model files: ['adapter_model.safetensors', 'adapter_config.json']...

💾 Google Drive Checkpoint:
   Path: /content/drive/MyDrive/codellama-coding-chat/checkpoint-250
   Status: ✅ EXISTS
   Size: 0.45 GB

🎯 Final Model:
   Path: ./codellama-coding-chat-final
   Status: ❌ NOT FOUND

📄 GitHub State File:
   Status: ✅ EXISTS
   Step: 250
   Status: paused_at_250

📊 SUMMARY
   ✅ Local Checkpoint (250)
   ✅ Drive Checkpoint
   ❌ Final Model
   ✅ State JSON
⚠️  SOME FILES MISSING - Check above.
